# Discovery


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

DATA_PATH = Path("/home/minu2k5/projects/Backdoor patching using machine unlearning/data/0_raw/IoTID20/IoT Network Intrusion Dataset.csv")

df = pd.read_csv(DATA_PATH)
df.columns = [c.strip() for c in df.columns]
print(f"Loaded dataset: {df.shape[0]} rows × {df.shape[1]} cols")
print(f"Duplicates: {df.duplicated().sum()} | Target class distribution:")
print(df["Cat"].value_counts().to_dict())

Loaded dataset: 625783 rows × 86 cols
Duplicates: 164087 | Target class distribution:
{'Mirai': 415677, 'Scan': 75265, 'DoS': 59391, 'Normal': 40073, 'MITM ARP Spoofing': 35377}


# Initial Cleaning

In [2]:
df_clean = df.copy()
df_clean = df_clean.drop_duplicates().reset_index(drop=True)

drop_cols = [c for c in ["Flow_ID", "Src_IP", "Dst_IP", "Src_Port", "Dst_Port", "Timestamp"] if c in df_clean.columns]
df_clean = df_clean.drop(columns=drop_cols, errors="ignore")
df_clean = df_clean.replace([np.inf, -np.inf], np.nan).dropna().reset_index(drop=True)

top_value_ratios = {col: df_clean[col].value_counts(normalize=True, dropna=False).iloc[0] for col in df_clean.columns}
constant_like_cols = [c for c in pd.Series(top_value_ratios)[lambda x: x > 0.999].index if c not in ["Label", "Cat", "Sub_Cat"]]
df_clean = df_clean.drop(columns=constant_like_cols, errors="ignore")
print(f"Cleaned: {df_clean.shape[0]} rows × {df_clean.shape[1]} cols")

Cleaned: 461373 rows × 64 cols


# Data Splitting and Scaling

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import QuantileTransformer, MinMaxScaler

TARGET_COLUMN = "Cat"
label_columns = [col for col in ["Label", "Cat", "Sub_Cat"] if col in df_clean.columns]
y = df_clean[TARGET_COLUMN].copy()
X = df_clean.drop(columns=label_columns, errors="ignore").copy()

RANDOM_STATE = 42
TEST_SIZE = 0.15
VAL_SIZE_WITHIN_TRAIN = 0.15 / (1 - TEST_SIZE)

X_train_val, X_test, y_train_val, y_test = train_test_split(X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=VAL_SIZE_WITHIN_TRAIN, random_state=RANDOM_STATE, stratify=y_train_val)

scaler_pipeline = Pipeline([
    ('quantile_transformer', QuantileTransformer(output_distribution='normal', random_state=RANDOM_STATE)),
    ('minmax_scaler', MinMaxScaler()),
])

X_train_scaled = pd.DataFrame(scaler_pipeline.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
X_val_scaled = pd.DataFrame(scaler_pipeline.transform(X_val), columns=X_val.columns, index=X_val.index)
X_test_scaled = pd.DataFrame(scaler_pipeline.transform(X_test), columns=X_test.columns, index=X_test.index)
print(f"Split: train={X_train.shape[0]}, val={X_val.shape[0]}, test={X_test.shape[0]} | Scaled: {X_train_scaled.shape}")

Split: train=322961, val=69206, test=69206 | Scaled: (322961, 61)


# Label Encoding


In [5]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)
y_val_encoded = label_encoder.transform(y_val)
y_test_encoded = label_encoder.transform(y_test)
print(f"Classes: {list(label_encoder.classes_)}")

Classes: ['DoS', 'MITM ARP Spoofing', 'Mirai', 'Normal', 'Scan']


# Prepared Data Summary

In [8]:
print(f"Data ready: {X_train_scaled.shape[1]} features, {len(label_encoder.classes_)} classes")
print("Train class distribution:")
train_dist = y_train.value_counts().sort_index()
for cls, count in train_dist.items():
    print(f"  {cls}: {count}")

Data ready: 61 features, 5 classes
Train class distribution:
  DoS: 41573
  MITM ARP Spoofing: 18104
  Mirai: 196545
  Normal: 27018
  Scan: 39721


# Model Training

We train a 3-layer MLP classifier with balanced cross-entropy loss to address class imbalance. The balanced class weights are computed as $w_c = \frac{n}{n_c \cdot k}$, where $n$ is total samples, $n_c$ is the count for class $c$, and $k$ is the number of classes. We employ early stopping on validation macro F1-score to prevent overfitting.

In [7]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
import numpy as np
import pandas as pd

device = "cuda"
train_loader = DataLoader(TensorDataset(torch.tensor(X_train_scaled.values, dtype=torch.float32), torch.tensor(y_train_encoded, dtype=torch.long)), batch_size=256, shuffle=True)
val_loader = DataLoader(TensorDataset(torch.tensor(X_val_scaled.values, dtype=torch.float32), torch.tensor(y_val_encoded, dtype=torch.long)), batch_size=512)
test_loader = DataLoader(TensorDataset(torch.tensor(X_test_scaled.values, dtype=torch.float32), torch.tensor(y_test_encoded, dtype=torch.long)), batch_size=512)

classes = np.arange(len(label_encoder.classes_))
balanced_weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_train_encoded)
class_weights_tensor = torch.tensor(balanced_weights, dtype=torch.float32, device=device)

print("Balanced Class Weights:")
display(pd.DataFrame({"class_name": label_encoder.classes_, "class_id": classes, "weight": balanced_weights}))

def build_model(input_dim, num_classes, random_state=42):
    torch.manual_seed(random_state)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(random_state)
    model = nn.Sequential(
        nn.Linear(input_dim, 256), nn.ReLU(), nn.Dropout(0.1),
        nn.Linear(256, 128), nn.ReLU(), nn.Dropout(0.1),
        nn.Linear(128, num_classes),
    ).to(device)
    return model

def evaluate(model, loader):
    model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for x, y in loader:
            logits = model(x.to(device))
            preds.append(logits.argmax(1).cpu())
            trues.append(y)
    y_pred = torch.cat(preds).numpy()
    y_true = torch.cat(trues).numpy()
    return {"acc": accuracy_score(y_true, y_pred), "macro_f1": f1_score(y_true, y_pred, average="macro"), "y_true": y_true, "y_pred": y_pred}

EPOCHS, PATIENCE, LR = 20, 5, 1e-3
print(f"Training: {EPOCHS} epochs, patience={PATIENCE}, lr={LR}")

model = build_model(X_train_scaled.shape[1], len(label_encoder.classes_), RANDOM_STATE)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
best_val_macro_f1, patience_counter = -1.0, 0
best_model_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_preds, train_trues = [], []
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        train_preds.append(logits.argmax(1).detach().cpu())
        train_trues.append(y.detach().cpu())
    
    train_acc = accuracy_score(torch.cat(train_trues).numpy(), torch.cat(train_preds).numpy())
    val_metrics = evaluate(model, val_loader)
    print(f"Epoch {epoch:02d} | train_acc={train_acc:.4f} | val_acc={val_metrics['acc']:.4f} | val_macro_f1={val_metrics['macro_f1']:.4f}")
    
    if val_metrics["macro_f1"] > best_val_macro_f1:
        best_val_macro_f1, patience_counter = val_metrics["macro_f1"], 0
        best_model_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"Early stopping at epoch {epoch}")
            break

model.load_state_dict(best_model_state)
test_metrics = evaluate(model, test_loader)
print(f"\n=== TEST RESULTS ===")
print(f"Accuracy: {test_metrics['acc']:.4f} | Macro F1: {test_metrics['macro_f1']:.4f}")
print("\nClassification Report:")
print(classification_report(test_metrics["y_true"], test_metrics["y_pred"], target_names=label_encoder.classes_, zero_division=0))
print("Confusion Matrix:")
display(pd.DataFrame(confusion_matrix(test_metrics["y_true"], test_metrics["y_pred"]), index=label_encoder.classes_, columns=label_encoder.classes_))

Balanced Class Weights:


,class_name,class_id,weight
0,DoS,0,1.553706
1,MITM ARP Spoofing,1,3.567841
2,Mirai,2,0.328638
3,Normal,3,2.390710
4,Scan,4,1.626147


Training: 20 epochs, patience=5, lr=0.001
Epoch 01 | train_acc=0.7119 | val_acc=0.7608 | val_macro_f1=0.7071
Epoch 02 | train_acc=0.7525 | val_acc=0.7790 | val_macro_f1=0.7259
Epoch 03 | train_acc=0.7634 | val_acc=0.7758 | val_macro_f1=0.7312
Epoch 04 | train_acc=0.7687 | val_acc=0.7774 | val_macro_f1=0.7335
Epoch 05 | train_acc=0.7725 | val_acc=0.7902 | val_macro_f1=0.7468
Epoch 06 | train_acc=0.7759 | val_acc=0.7779 | val_macro_f1=0.7283
Epoch 07 | train_acc=0.7791 | val_acc=0.7917 | val_macro_f1=0.7484
Epoch 08 | train_acc=0.7817 | val_acc=0.7930 | val_macro_f1=0.7508
Epoch 09 | train_acc=0.7846 | val_acc=0.7946 | val_macro_f1=0.7512
Epoch 10 | train_acc=0.7864 | val_acc=0.7911 | val_macro_f1=0.7433
Epoch 11 | train_acc=0.7879 | val_acc=0.7910 | val_macro_f1=0.7471
Epoch 12 | train_acc=0.7885 | val_acc=0.7977 | val_macro_f1=0.7548
Epoch 13 | train_acc=0.7900 | val_acc=0.7958 | val_macro_f1=0.7487
Epoch 14 | train_acc=0.7918 | val_acc=0.7968 | val_macro_f1=0.7494
Epoch 15 | train_acc

,DoS,MITM ARP Spoofing,Mirai,Normal,Scan
DoS,8898,3,2,5,0
MITM ARP Spoofing,0,2431,40,24,1384
Mirai,1,4578,31626,198,5714
Normal,1,114,88,5023,564
Scan,0,759,71,93,7589
